# NIFTY Implied-Volatility Imputation
**Finance Club, IIT Roorkee — Open Project 2026**

Pipeline: a per-timestamp volatility-smile fit gives the bulk of each estimate, and a learned residual model (HistGradientBoosting + bagged HGBM, averaged) corrects what the smile misses.

$$\widehat{\sigma}(t,K) \;=\; \underbrace{\hat\sigma_{\text{smile}}(t,K)}_{\text{IDW-weighted quadratic over neighbouring strikes}} \;+\; \underbrace{\bar r_{\text{boost}}(t,K)}_{\text{mean of single + bagged HGBM}}$$

Run top-to-bottom with `dataset.csv` in the working directory. Requires `numpy`, `pandas`, `scikit-learn`. Every estimator is seeded — re-running yields a bit-identical `submission.csv`.

## 1 · Problem framing

975 five-minute snapshots over Jan 7–27, 2026, one expiry (27 Jan 26), 28 strikes split into 14 calls and 14 puts. `datetime` and `underlying_price` are complete; ~20% of the IV cells are NaN and are what we score on (MSE).

Key facts the model is built around:

1. **One expiry**, so no term structure — just a smile evolving over time.
2. **Disjoint strike support for CE and PE** in this snapshot. They are *not* interchangeable observations of the same underlying IV at this expiry; we treat the two wings as separate functions.
3. **Strike-localised dropout.** Missingness is concentrated at OTM strikes (low liquidity); same-timestamp cross-strike interpolation is therefore the strongest signal, with temporal continuity as a secondary correction.
4. **Single expiry + dense strike grid** ⇒ we *can* fit a smile per timestamp; we do not need a global parametric surface.

In [1]:
import numpy as np
import pandas as pd
import warnings
warnings.filterwarnings('ignore')
from sklearn.ensemble import HistGradientBoostingRegressor, BaggingRegressor

SEED = 0   # single seed anchors every random component below

## 2 · Load & inventory

Tree models, so we do no scaling/encoding. We keep observed cells as floats and missing cells as `NaN`; the latter are what we ultimately predict. Strikes are parsed from the ticker (`NIFTY27JAN26`**`25200`**`CE` → `25200`).

In [2]:
data = pd.read_csv('dataset.csv')
opt_cols  = [c for c in data.columns if c not in ('datetime', 'underlying_price')]
call_cols = [c for c in opt_cols if c.endswith('CE')]
put_cols  = [c for c in opt_cols if c.endswith('PE')]
get_strike = lambda c: int(c[12:-2])

print(f'{len(data)} timestamps, {len(opt_cols)} IV columns ({len(call_cols)} CE / {len(put_cols)} PE)')
print(f'Missing IV cells to predict: {data[opt_cols].isna().sum().sum()}')

975 timestamps, 28 IV columns (14 CE / 14 PE)
Missing IV cells to predict: 5460


## 3 · Volatility-smile estimator

`smile_fit` returns the IV at a target strike from the *same-timestamp* smile only (no temporal lookup, no future leakage). The mechanics:

- Gather up to four observed strikes immediately below and four immediately above the target, restricted to the same option type.
- Fit a polynomial of degree `min(2, n−1)` weighted by `1 / |k − k_target|` (with a 50-point distance floor to avoid division blow-up).
- Sanity-check the estimate against the local IV band `[½·min, 2·max]` of those support points; if it escapes that band, fall back to a linear blend between the two nearest neighbours.
- At the wings (one-sided only), use a low-order extrapolation over the 2–3 nearest same-side points.

In [3]:
def smile_fit(row, target_col, peers, is_call):
    """Same-timestamp smile estimate of IV at the target strike."""
    k_target = get_strike(target_col)
    pts = sorted(
        [(get_strike(c), float(row[c])) for c in peers
         if c != target_col and pd.notna(row[c]) and float(row[c]) > 0],
        key=lambda x: x[0],
    )
    if len(pts) < 2:
        return np.nan

    lower = sorted([(k, v) for k, v in pts if k < k_target], key=lambda x: -x[0])
    upper = sorted([(k, v) for k, v in pts if k > k_target], key=lambda x:  x[0])

    if lower and upper:
        support = lower[:4] + upper[:4]
        support.sort(key=lambda x: x[0])
        ks = np.array([p[0] for p in support])
        vs = np.array([p[1] for p in support])
        try:
            d = np.abs(ks - k_target).astype(float)
            d[d < 50] = 50
            coeffs = np.polyfit(ks, vs, min(2, len(ks) - 1), w=1.0 / d)
            est = float(np.polyval(coeffs, k_target))
            band_lo = min(vs[np.argmin(np.abs(ks - k_target))], vs.min())
            band_hi = max(vs[np.argmax(np.abs(ks - k_target))], vs.max())
            if est < band_lo * 0.5 or est > band_hi * 2.0:
                lK, lIV = lower[0]; rK, rIV = upper[0]
                return lIV + (k_target - lK) / (rK - lK) * (rIV - lIV)
            return est
        except Exception:
            lK, lIV = lower[0]; rK, rIV = upper[0]
            return lIV + (k_target - lK) / (rK - lK) * (rIV - lIV)

    one_side = sorted(lower if lower else upper, key=lambda x: abs(x[0] - k_target))
    side_ks  = [s[0] for s in one_side]
    out_of_chain = (k_target > max(side_ks)) if is_call else (k_target < min(side_ks))
    near = one_side[:3] if out_of_chain else one_side[:2]
    near.sort(key=lambda x: x[0])
    ks = [p[0] for p in near]; vs = [p[1] for p in near]
    try:
        return float(np.polyval(np.polyfit(ks, vs, min(1, len(ks) - 1)), k_target))
    except Exception:
        return near[0][1]

## 4 · Feature matrix

For every (timestamp, strike) cell we emit a 13-dimensional feature row. Every feature is either same-timestamp or *strictly past* relative to that cell — no entry uses a future value of the cell itself.

| # | Feature | What it captures |
|---|---------|-------------------|
| 1 | `log(K / S)` | log-moneyness |
| 2 | `K` | raw strike |
| 3 | `is_call` | 1 for CE, 0 for PE |
| 4 | `S` | spot |
| 5 | `t / T` | normalised time index |
| 6 | `median(IV obs at t)` | smile level |
| 7 | `std(IV obs at t)` | smile dispersion |
| 8 | `smile_fit(...)` | the smile estimate itself |
| 9 | last observed IV in same column | temporal carry-forward |
| 10–11 | nearest *lower* neighbour: IV, strike-gap | local left context |
| 12–13 | nearest *upper* neighbour: IV, strike-gap | local right context |

We also return the smile estimate (`smile_anchor`) so the model trains on the **residual** `actual − smile_anchor` and we add the smile back at predict time.

In [4]:
def assemble_features(frame):
    """Returns (X, address_list, y, smile_anchors). Cells with missing IV have y = NaN."""
    n_rows = len(frame)
    spot   = frame['underlying_price'].values
    rows, ys, addr, anchors = [], [], [], []

    for i in range(n_rows):
        snap = frame.iloc[i]
        S    = spot[i]
        all_obs   = [snap[c] for c in opt_cols if pd.notna(snap[c]) and snap[c] > 0]
        iv_mid    = float(np.median(all_obs)) if all_obs else 0.15
        iv_spread = float(np.std(all_obs))    if all_obs else 0.0

        for col in opt_cols:
            is_call = col.endswith('CE')
            peers   = call_cols if is_call else put_cols
            K       = get_strike(col)

            smile  = smile_fit(snap, col, peers, is_call)
            anchor = smile if pd.notna(smile) else iv_mid

            obs = sorted([(get_strike(c), float(snap[c])) for c in peers
                          if c != col and pd.notna(snap[c]) and snap[c] > 0])
            lo  = [o for o in obs if o[0] < K]
            hi  = [o for o in obs if o[0] > K]
            ln  = lo[-1] if lo else (np.nan, np.nan)
            hn  = hi[0]  if hi else (np.nan, np.nan)

            hist  = frame[col].iloc[:i]
            l_idx = hist.last_valid_index()
            carry = float(frame[col].iloc[l_idx]) if l_idx is not None else iv_mid

            rows.append([
                np.log(K / S), K, 1.0 if is_call else 0.0, S, i / n_rows, iv_mid, iv_spread,
                anchor, carry,
                ln[1] if not np.isnan(ln[1]) else iv_mid, (K - ln[0]) if not np.isnan(ln[0]) else 999,
                hn[1] if not np.isnan(hn[1]) else iv_mid, (hn[0] - K) if not np.isnan(hn[0]) else 999,
            ])
            addr.append((i, col))
            anchors.append(anchor)
            ys.append(float(snap[col]) if pd.notna(snap[col]) else np.nan)

    return np.array(rows), addr, np.array(ys), np.array(anchors)

## 5 · Residual learner — single HGBM averaged with a bagged HGBM

**Why train on residuals, not on the IV directly?** The smile fit already explains most of the cross-strike variation; the boosted model only has to capture the *remainder*. Residual learning is dramatically easier to fit and dramatically less prone to overfitting the noise in observed IVs.

**Why average two HGBMs?** The single HGBM tends to be sharp; the bagged HGBM (10 estimators, 80% rows + 80% features) is smoother and more robust at the wings where data is thin. Averaging recovers most of the sharpness while damping the wing variance. This was the configuration that beat both the single and the bagged model alone on the public leaderboard.

**The strict rule:** training pairs are observed cells that are not themselves a target this round. The held-out / missing cells are never in the training set.

In [5]:
def _make_hgbm():
    return HistGradientBoostingRegressor(
        max_iter=400, learning_rate=0.05,
        l2_regularization=1.0, min_samples_leaf=20,
        random_state=SEED,
    )

def train_impute(frame, target_mask):
    """Train single+bagged HGBM on observed residuals; return {(i, col): IV} for target cells."""
    X, addr, y, anchors = assemble_features(frame)
    observed = ~np.isnan(y)
    targeted = np.array([target_mask.loc[i, c] for (i, c) in addr])
    train    = observed & (~targeted)

    Xtr, ytr = X[train], (y - anchors)[train]

    r_single = _make_hgbm().fit(Xtr, ytr).predict(X)
    r_bagged = BaggingRegressor(
        estimator=_make_hgbm(), n_estimators=10,
        max_samples=0.8, max_features=0.8,
        random_state=SEED, n_jobs=-1,
    ).fit(Xtr, ytr).predict(X)

    out = {}
    for j, (i, c) in enumerate(addr):
        if target_mask.loc[i, c]:
            v1 = max(0.005, float(anchors[j] + r_single[j]))
            v2 = max(0.005, float(anchors[j] + r_bagged[j]))
            out[(i, c)] = 0.5 * (v1 + v2)
    return out

## 6 · Self-validation (sanity check)

Hide a random 15% of the *observed* cells, run the full pipeline against the masked frame, and score against the true values that were hidden. This is leak-free with respect to the model: held-out cells are excluded from training.

**Honest caveat:** the cells we *can* hide are the liquid ones; the cells the leaderboard actually scores are the illiquid (already-missing) ones. So this number tends to under-estimate the true test MSE for any high-capacity model. Use it for sanity, not for model selection — the public leaderboard was the decision metric.

In [6]:
def holdout_validate(frac=0.15, seed=SEED):
    rng  = np.random.default_rng(seed)
    mask = pd.DataFrame(False, index=data.index, columns=opt_cols)
    for col in opt_cols:
        idx_obs = data.index[data[col].notna()].to_numpy()
        chosen  = rng.choice(idx_obs, size=max(1, int(frac * len(idx_obs))), replace=False)
        mask.loc[chosen, col] = True

    work = data.copy()
    work[opt_cols] = data[opt_cols].mask(mask)
    preds = train_impute(work, mask)
    mse   = float(np.mean([(preds[(i, c)] - data.loc[i, c]) ** 2 for (i, c) in preds]))
    print(f'Self-validation MSE ({len(preds)} held-out liquid cells, seed {seed}): {mse:.6f}')
    return mse

_ = holdout_validate()

Self-validation MSE (3262 held-out liquid cells, seed 0): 0.000034


## 7 · Final fit on all observed cells, then write the submission

In [7]:
to_fill     = data[opt_cols].isna()
predictions = train_impute(data, to_fill)

completed = data.copy()
for (i, c), v in predictions.items():
    completed.at[i, c] = v
print(f'Filled {len(predictions)} cells; remaining NaN: {completed[opt_cols].isna().sum().sum()}')
completed.to_csv('filled_dataset.csv', index=False)

def build_submission(filled_path, out_path='submission.csv'):
    src    = pd.read_csv('dataset.csv')
    filled = pd.read_csv(filled_path)
    rows = []
    for col in [c for c in src.columns if c != 'datetime']:
        for i in src.index[src[col].isna()]:
            dt = src.loc[i, 'datetime']
            rows.append({'id': f'{dt}||{col}', 'value': filled.loc[i, col]})
    sol = pd.DataFrame(rows, columns=['id', 'value']).sort_values('id').reset_index(drop=True)
    sol.to_csv(out_path, index=False)
    print(f'Saved {out_path}  ({len(sol)} rows)')
    return sol

submission = build_submission('filled_dataset.csv', 'submission.csv')
submission.head()

Filled 5460 cells; remaining NaN: 0
Saved submission.csv  (5460 rows)


,id,value
0,07-01-2026 09:15||NIFTY27JAN2624100PE,0.163508
1,07-01-2026 09:15||NIFTY27JAN2625500CE,0.113829
2,07-01-2026 09:15||NIFTY27JAN2625800CE,0.101524
3,07-01-2026 09:20||NIFTY27JAN2624000PE,0.170319
4,07-01-2026 09:20||NIFTY27JAN2624200PE,0.159817


## 8 · Output integrity checks

Row count matches the number of missing cells; no NaNs in the submission; all IVs are positive; every `id` follows the `datetime||ticker` format.

In [8]:
assert len(submission) == int(data[opt_cols].isna().sum().sum()), 'row count must equal #missing cells'
assert submission['value'].isna().sum() == 0, 'no NaNs allowed'
assert (submission['value'] > 0).all(), 'IV must be positive'
assert submission['id'].str.contains(r'\|\|').all(), 'id must be datetime||ticker'
print('All checks passed.')
print(f"rows={len(submission)}  value range [{submission['value'].min():.4f}, {submission['value'].max():.4f}]  median {submission['value'].median():.4f}")

All checks passed.
rows=5460  value range [0.0293, 5.7796]  median 0.1311
